In [1]:
from pathlib import Path
import zipfile
import hashlib
import re

import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_ZIP = PROJECT_ROOT / "data" / "raw" / "archive.zip"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "data_cleaning"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw zip exists:", RAW_ZIP.exists())
print("Raw zip path:", RAW_ZIP)

Project root: /Users/charan/Documents/supermarket-price-direction
Raw zip exists: True
Raw zip path: /Users/charan/Documents/supermarket-price-direction/data/raw/archive.zip


In [3]:
with zipfile.ZipFile(RAW_ZIP, "r") as z:
    zip_files = z.namelist()

csv_files = [
    f for f in zip_files
    if f.endswith(".csv") and not f.startswith("__MACOSX")
]

csv_files

['All_Data_ASDA.csv',
 'All_Data_Aldi.csv',
 'All_Data_Morrisons.csv',
 'All_Data_Sains.csv',
 'All_Data_Tesco.csv']

In [4]:
dataframes = {}

with zipfile.ZipFile(RAW_ZIP, "r") as z:
    for file in csv_files:
        retailer_name = Path(file).stem.replace("All_Data_", "")
        dataframes[retailer_name] = pd.read_csv(z.open(file), low_memory=False)

for retailer, df in dataframes.items():
    print(retailer, df.shape)

ASDA (2456414, 8)
Aldi (464863, 8)
Morrisons (1794065, 8)
Sains (2600289, 8)
Tesco (2213611, 8)


In [5]:
combined_raw = []

for retailer, df in dataframes.items():
    temp = df.copy()
    temp["retailer_file"] = retailer
    combined_raw.append(temp)

raw = pd.concat(combined_raw, ignore_index=True)

print("Raw rows:", len(raw))
print("Raw columns:", raw.columns.tolist())

raw.head()

Raw rows: 9529242
Raw columns: ['supermarket', 'prices_(£)', 'prices_unit_(£)', 'unit', 'names', 'date', 'category', 'own_brand', 'retailer_file']


,supermarket,prices_(£),prices_unit_(£),unit,names,date,category,own_brand,retailer_file
0,ASDA,4.75,19.8,kg,Tassimo Cadbury Hot Chocolate Pods x 8,20240413,drinks,False,ASDA
1,ASDA,2.00,26.7,kg,Taylors of Harrogate Hot Lava Java Coffee Bags,20240413,drinks,False,ASDA
2,ASDA,5.00,20.8,kg,Tassimo Limited Edition Cadbury Orange Hot Cho...,20240413,drinks,False,ASDA
3,ASDA,3.50,15.4,kg,ASDA Extra Special Brazilian Ground Coffee,20240413,drinks,True,ASDA
4,ASDA,3.50,15.4,kg,ASDA Extra Special Espresso Coffee Beans,20240413,drinks,True,ASDA


In [6]:
clean = raw.copy()

clean = clean.rename(columns={
    "supermarket": "retailer",
    "prices_(£)": "price",
    "prices_unit_(£)": "price_per_unit",
    "names": "product_name",
    "own_brand": "own_brand_flag"
})

clean["date"] = pd.to_datetime(
    clean["date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

clean["retailer"] = clean["retailer"].astype(str).str.strip()
clean["category"] = clean["category"].astype(str).str.strip()
clean["product_name"] = clean["product_name"].astype(str).str.strip()
clean["unit"] = clean["unit"].astype(str).str.strip()

clean["price"] = pd.to_numeric(clean["price"], errors="coerce")
clean["price_per_unit"] = pd.to_numeric(clean["price_per_unit"], errors="coerce")

clean.head()

,retailer,price,price_per_unit,unit,product_name,date,category,own_brand_flag,retailer_file
0,ASDA,4.75,19.8,kg,Tassimo Cadbury Hot Chocolate Pods x 8,2024-04-13,drinks,False,ASDA
1,ASDA,2.00,26.7,kg,Taylors of Harrogate Hot Lava Java Coffee Bags,2024-04-13,drinks,False,ASDA
2,ASDA,5.00,20.8,kg,Tassimo Limited Edition Cadbury Orange Hot Cho...,2024-04-13,drinks,False,ASDA
3,ASDA,3.50,15.4,kg,ASDA Extra Special Brazilian Ground Coffee,2024-04-13,drinks,True,ASDA
4,ASDA,3.50,15.4,kg,ASDA Extra Special Espresso Coffee Beans,2024-04-13,drinks,True,ASDA


In [7]:
clean["own_brand_flag_raw"] = clean["own_brand_flag"]

clean["own_brand_flag"] = (
    clean["own_brand_flag"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False
    })
)

own_brand_check = (
    clean.groupby(["retailer", "own_brand_flag"], dropna=False)
    .size()
    .reset_index(name="rows")
)

own_brand_check

,retailer,own_brand_flag,rows
0,ASDA,False,1741841
1,ASDA,True,714547
2,ASDA,NaN,26
3,Aldi,False,385723
4,Aldi,True,79140
5,Morrisons,False,1357322
6,Morrisons,True,436743
7,Sains,False,2006487
8,Sains,True,593802
9,Tesco,False,1639834


In [8]:
standardised = clean[
    [
        "retailer",
        "category",
        "date",
        "product_name",
        "price",
        "price_per_unit",
        "unit",
        "own_brand_flag",
        "own_brand_flag_raw",
        "retailer_file"
    ]
].copy()

print("Standardised shape:", standardised.shape)
standardised.head()

Standardised shape: (9529242, 10)


,retailer,category,date,product_name,price,price_per_unit,unit,own_brand_flag,own_brand_flag_raw,retailer_file
0,ASDA,drinks,2024-04-13,Tassimo Cadbury Hot Chocolate Pods x 8,4.75,19.8,kg,False,False,ASDA
1,ASDA,drinks,2024-04-13,Taylors of Harrogate Hot Lava Java Coffee Bags,2.00,26.7,kg,False,False,ASDA
2,ASDA,drinks,2024-04-13,Tassimo Limited Edition Cadbury Orange Hot Cho...,5.00,20.8,kg,False,False,ASDA
3,ASDA,drinks,2024-04-13,ASDA Extra Special Brazilian Ground Coffee,3.50,15.4,kg,True,True,ASDA
4,ASDA,drinks,2024-04-13,ASDA Extra Special Espresso Coffee Beans,3.50,15.4,kg,True,True,ASDA


In [9]:
def normalise_product_name(name):
    if pd.isna(name):
        return np.nan
    
    name = str(name).lower()
    name = re.sub(r"[^a-z0-9]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

standardised["product_name_normalised"] = standardised["product_name"].apply(normalise_product_name)

standardised[["retailer", "product_name", "product_name_normalised"]].head()

,retailer,product_name,product_name_normalised
0,ASDA,Tassimo Cadbury Hot Chocolate Pods x 8,tassimo cadbury hot chocolate pods x 8
1,ASDA,Taylors of Harrogate Hot Lava Java Coffee Bags,taylors of harrogate hot lava java coffee bags
2,ASDA,Tassimo Limited Edition Cadbury Orange Hot Cho...,tassimo limited edition cadbury orange hot cho...
3,ASDA,ASDA Extra Special Brazilian Ground Coffee,asda extra special brazilian ground coffee
4,ASDA,ASDA Extra Special Espresso Coffee Beans,asda extra special espresso coffee beans


In [10]:
product_lookup = (
    standardised[["retailer", "product_name_normalised"]]
    .drop_duplicates()
    .sort_values(["retailer", "product_name_normalised"])
    .reset_index(drop=True)
)

product_lookup["product_number"] = (
    product_lookup
    .groupby("retailer")
    .cumcount() + 1
)

product_lookup["retailer_slug"] = (
    product_lookup["retailer"]
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

product_lookup["product_id"] = (
    product_lookup["retailer_slug"]
    + "_"
    + product_lookup["product_number"].astype(str).str.zfill(6)
)

product_lookup = product_lookup[
    ["product_id", "retailer", "product_name_normalised", "product_number"]
]

print("Product lookup shape:", product_lookup.shape)
product_lookup.head()

Product lookup shape: (130296, 4)


,product_id,retailer,product_name_normalised,product_number
0,asda_000001,ASDA,1000 stories zinfandel 75cl,1
1,asda_000002,ASDA,1001 carpet fresh pet thai orchid passionfruit...,2
2,asda_000003,ASDA,151 2x3g super glue,3
3,asda_000004,ASDA,151 all purpose super glue,4
4,asda_000005,ASDA,151 gorilla filler 500ml,5


In [11]:
standardised = standardised.merge(
    product_lookup[["product_id", "retailer", "product_name_normalised"]],
    on=["retailer", "product_name_normalised"],
    how="left",
    validate="many_to_one"
)

print("Rows after product_id merge:", len(standardised))
print("Missing product_id:", standardised["product_id"].isna().sum())

standardised.head()

Rows after product_id merge: 9529242
Missing product_id: 0


,retailer,category,date,product_name,price,price_per_unit,unit,own_brand_flag,own_brand_flag_raw,retailer_file,product_name_normalised,product_id
0,ASDA,drinks,2024-04-13,Tassimo Cadbury Hot Chocolate Pods x 8,4.75,19.8,kg,False,False,ASDA,tassimo cadbury hot chocolate pods x 8,asda_029531
1,ASDA,drinks,2024-04-13,Taylors of Harrogate Hot Lava Java Coffee Bags,2.00,26.7,kg,False,False,ASDA,taylors of harrogate hot lava java coffee bags,asda_029570
2,ASDA,drinks,2024-04-13,Tassimo Limited Edition Cadbury Orange Hot Cho...,5.00,20.8,kg,False,False,ASDA,tassimo limited edition cadbury orange hot cho...,asda_029551
3,ASDA,drinks,2024-04-13,ASDA Extra Special Brazilian Ground Coffee,3.50,15.4,kg,True,True,ASDA,asda extra special brazilian ground coffee,asda_002547
4,ASDA,drinks,2024-04-13,ASDA Extra Special Espresso Coffee Beans,3.50,15.4,kg,True,True,ASDA,asda extra special espresso coffee beans,asda_002645


In [12]:
standardised = standardised[
    [
        "product_id",
        "retailer",
        "category",
        "date",
        "product_name",
        "product_name_normalised",
        "price",
        "price_per_unit",
        "unit",
        "own_brand_flag",
        "own_brand_flag_raw",
        "retailer_file"
    ]
].copy()

standardised.head()

,product_id,retailer,category,date,product_name,product_name_normalised,price,price_per_unit,unit,own_brand_flag,own_brand_flag_raw,retailer_file
0,asda_029531,ASDA,drinks,2024-04-13,Tassimo Cadbury Hot Chocolate Pods x 8,tassimo cadbury hot chocolate pods x 8,4.75,19.8,kg,False,False,ASDA
1,asda_029570,ASDA,drinks,2024-04-13,Taylors of Harrogate Hot Lava Java Coffee Bags,taylors of harrogate hot lava java coffee bags,2.00,26.7,kg,False,False,ASDA
2,asda_029551,ASDA,drinks,2024-04-13,Tassimo Limited Edition Cadbury Orange Hot Cho...,tassimo limited edition cadbury orange hot cho...,5.00,20.8,kg,False,False,ASDA
3,asda_002547,ASDA,drinks,2024-04-13,ASDA Extra Special Brazilian Ground Coffee,asda extra special brazilian ground coffee,3.50,15.4,kg,True,True,ASDA
4,asda_002645,ASDA,drinks,2024-04-13,ASDA Extra Special Espresso Coffee Beans,asda extra special espresso coffee beans,3.50,15.4,kg,True,True,ASDA


In [13]:
product_lookup.to_csv(
    RESULTS_DIR / "product_lookup.csv",
    index=False
)

print("Saved:", RESULTS_DIR / "product_lookup.csv")

Saved: /Users/charan/Documents/supermarket-price-direction/results/data_cleaning/product_lookup.csv


In [14]:
    key_cols = ["product_id", "retailer", "date"]

duplicate_mask = standardised.duplicated(subset=key_cols, keep=False)

duplicate_before_summary = (
    standardised.loc[duplicate_mask]
    .groupby("retailer")
    .size()
    .reset_index(name="duplicated_rows_before")
)

retailer_rows_before = (
    standardised.groupby("retailer")
    .size()
    .reset_index(name="rows_before")
)

duplicate_before_summary = retailer_rows_before.merge(
    duplicate_before_summary,
    on="retailer",
    how="left"
).fillna({"duplicated_rows_before": 0})

duplicate_before_summary["duplicated_rows_before"] = duplicate_before_summary["duplicated_rows_before"].astype(int)

duplicate_before_summary["duplicated_rows_before_pct"] = (
    duplicate_before_summary["duplicated_rows_before"]
    / duplicate_before_summary["rows_before"]
    * 100
).round(4)

duplicate_before_summary

IndentationError: unexpected indent (3500555917.py, line 1)

In [15]:
key_cols = ["product_id", "retailer", "date"]

In [16]:
key_cols = ["product_id", "retailer", "date"]

duplicate_mask = standardised.duplicated(subset=key_cols, keep=False)

duplicate_before_summary = (
    standardised.loc[duplicate_mask]
    .groupby("retailer")
    .size()
    .reset_index(name="duplicated_rows_before")
)

retailer_rows_before = (
    standardised.groupby("retailer")
    .size()
    .reset_index(name="rows_before")
)

duplicate_before_summary = retailer_rows_before.merge(
    duplicate_before_summary,
    on="retailer",
    how="left"
).fillna({"duplicated_rows_before": 0})

duplicate_before_summary["duplicated_rows_before"] = (
    duplicate_before_summary["duplicated_rows_before"].astype(int)
)

duplicate_before_summary["duplicated_rows_before_pct"] = (
    duplicate_before_summary["duplicated_rows_before"]
    / duplicate_before_summary["rows_before"]
    * 100
).round(4)

duplicate_before_summary

,retailer,rows_before,duplicated_rows_before,duplicated_rows_before_pct
0,ASDA,2456414,4496,0.1830
1,Aldi,464863,320,0.0688
2,Morrisons,1794065,8760,0.4883
3,Sains,2600289,906,0.0348
4,Tesco,2213611,48810,2.2050


In [17]:
standardised = standardised.reset_index(drop=False).rename(columns={"index": "raw_row_id"})

useful_fields = [
    "category",
    "product_name",
    "price",
    "price_per_unit",
    "unit",
    "own_brand_flag"
]

standardised["completeness_score"] = (
    standardised[useful_fields]
    .notna()
    .sum(axis=1)
)

standardised_sorted = standardised.sort_values(
    by=["product_id", "retailer", "date", "completeness_score", "raw_row_id"],
    ascending=[True, True, True, False, True]
)

duplicate_keep_mask = ~standardised_sorted.duplicated(
    subset=["product_id", "retailer", "date"],
    keep="first"
)

duplicate_drop_log = standardised_sorted.loc[
    ~duplicate_keep_mask,
    [
        "raw_row_id",
        "product_id",
        "retailer",
        "date",
        "product_name",
        "price",
        "price_per_unit",
        "unit",
        "own_brand_flag",
        "completeness_score"
    ]
].copy()

deduplicated = standardised_sorted.loc[duplicate_keep_mask].copy()

print("Rows before duplicate removal:", len(standardised))
print("Rows dropped as duplicate keys:", len(duplicate_drop_log))
print("Rows after duplicate removal:", len(deduplicated))

Rows before duplicate removal: 9529242
Rows dropped as duplicate keys: 31651
Rows after duplicate removal: 9497591


In [18]:
duplicate_drop_log.to_csv(
    RESULTS_DIR / "duplicate_drop_log.csv",
    index=False
)

duplicate_before_summary.to_csv(
    RESULTS_DIR / "duplicate_before_summary.csv",
    index=False
)

print("Saved duplicate drop log:", RESULTS_DIR / "duplicate_drop_log.csv")
print("Saved duplicate before summary:", RESULTS_DIR / "duplicate_before_summary.csv")

Saved duplicate drop log: /Users/charan/Documents/supermarket-price-direction/results/data_cleaning/duplicate_drop_log.csv
Saved duplicate before summary: /Users/charan/Documents/supermarket-price-direction/results/data_cleaning/duplicate_before_summary.csv


In [19]:
duplicate_after_count = deduplicated.duplicated(
    subset=["product_id", "retailer", "date"],
    keep=False
).sum()

print("Duplicate product_id-retailer-date rows after cleaning:", duplicate_after_count)

Duplicate product_id-retailer-date rows after cleaning: 0


In [20]:
product_name_mapping = (
    standardised[
        [
            "retailer",
            "product_id",
            "product_name",
            "product_name_normalised"
        ]
    ]
    .drop_duplicates()
    .sort_values(["retailer", "product_id", "product_name"])
    .reset_index(drop=True)
)

product_name_mapping.to_csv(
    RESULTS_DIR / "product_name_mapping.csv",
    index=False
)

print("Product-name mapping rows:", len(product_name_mapping))
print("Saved:", RESULTS_DIR / "product_name_mapping.csv")

product_name_mapping.head()

Product-name mapping rows: 130633
Saved: /Users/charan/Documents/supermarket-price-direction/results/data_cleaning/product_name_mapping.csv


,retailer,product_id,product_name,product_name_normalised
0,ASDA,asda_000001,1000 Stories Zinfandel 75cl,1000 stories zinfandel 75cl
1,ASDA,asda_000002,1001 Carpet Fresh Pet Thai Orchid & Passionfru...,1001 carpet fresh pet thai orchid passionfruit...
2,ASDA,asda_000003,151 2X3G Super Glue,151 2x3g super glue
3,ASDA,asda_000004,151 All Purpose Super Glue,151 all purpose super glue
4,ASDA,asda_000005,151 Gorilla Filler 500ml,151 gorilla filler 500ml


In [21]:
deduplicated = deduplicated.sort_values(
    ["retailer", "product_id", "date"]
).copy()

deduplicated["previous_capture_date"] = (
    deduplicated
    .groupby(["retailer", "product_id"])["date"]
    .shift(1)
)

deduplicated["days_since_previous_capture"] = (
    deduplicated["date"] - deduplicated["previous_capture_date"]
).dt.days

deduplicated["capture_gap_before_row"] = (
    deduplicated["days_since_previous_capture"] > 1
)

gap_summary = (
    deduplicated
    .groupby("retailer")
    .agg(
        rows=("product_id", "size"),
        rows_after_capture_gap=("capture_gap_before_row", "sum")
    )
    .reset_index()
)

gap_summary["rows_after_capture_gap_pct"] = (
    gap_summary["rows_after_capture_gap"]
    / gap_summary["rows"]
    * 100
).round(4)

gap_summary

,retailer,rows,rows_after_capture_gap,rows_after_capture_gap_pct
0,ASDA,2454166,157059,6.3997
1,Aldi,464703,27990,6.0232
2,Morrisons,1789685,161427,9.0199
3,Sains,2599836,100734,3.8746
4,Tesco,2189201,208735,9.5348


In [22]:
gap_summary.to_csv(
    RESULTS_DIR / "capture_gap_flag_summary.csv",
    index=False
)

print("Saved:", RESULTS_DIR / "capture_gap_flag_summary.csv")

Saved: /Users/charan/Documents/supermarket-price-direction/results/data_cleaning/capture_gap_flag_summary.csv


In [23]:
tidy = deduplicated[
    [
        "product_id",
        "retailer",
        "category",
        "date",
        "product_name",
        "product_name_normalised",
        "price",
        "price_per_unit",
        "unit",
        "own_brand_flag",
        "capture_gap_before_row",
        "previous_capture_date",
        "days_since_previous_capture"
    ]
].copy()

tidy = tidy.sort_values(["retailer", "product_id", "date"]).reset_index(drop=True)

print("Tidy shape:", tidy.shape)
tidy.head()

Tidy shape: (9497591, 13)


,product_id,retailer,category,date,product_name,product_name_normalised,price,price_per_unit,unit,own_brand_flag,capture_gap_before_row,previous_capture_date,days_since_previous_capture
0,asda_000001,ASDA,drinks,2024-01-09,1000 Stories Zinfandel 75cl,1000 stories zinfandel 75cl,15.0,199.95,l,False,False,NaT,NaN
1,asda_000001,ASDA,drinks,2024-01-10,1000 Stories Zinfandel 75cl,1000 stories zinfandel 75cl,15.0,199.95,l,False,False,2024-01-09,1.0
2,asda_000001,ASDA,drinks,2024-01-11,1000 Stories Zinfandel 75cl,1000 stories zinfandel 75cl,15.0,199.95,l,False,False,2024-01-10,1.0
3,asda_000001,ASDA,drinks,2024-01-12,1000 Stories Zinfandel 75cl,1000 stories zinfandel 75cl,15.0,199.95,l,False,False,2024-01-11,1.0
4,asda_000001,ASDA,drinks,2024-01-13,1000 Stories Zinfandel 75cl,1000 stories zinfandel 75cl,15.0,199.95,l,False,False,2024-01-12,1.0


In [24]:
rows_in = len(raw)
rows_removed_duplicates = len(duplicate_drop_log)
rows_out = len(tidy)

reconciliation = pd.DataFrame([
    {
        "metric": "rows_in_raw",
        "rows": rows_in
    },
    {
        "metric": "rows_removed_duplicate_keys",
        "rows": rows_removed_duplicates
    },
    {
        "metric": "rows_out_tidy",
        "rows": rows_out
    },
    {
        "metric": "rows_in_minus_removed",
        "rows": rows_in - rows_removed_duplicates
    }
])

reconciliation["matches_expected"] = reconciliation["rows_out_tidy" if False else "rows"] == reconciliation["rows"]

print("Rows in:", rows_in)
print("Rows removed:", rows_removed_duplicates)
print("Rows out:", rows_out)
print("Check rows_in - rows_removed == rows_out:", rows_in - rows_removed_duplicates == rows_out)

reconciliation

Rows in: 9529242
Rows removed: 31651
Rows out: 9497591
Check rows_in - rows_removed == rows_out: True


,metric,rows,matches_expected
0,rows_in_raw,9529242,True
1,rows_removed_duplicate_keys,31651,True
2,rows_out_tidy,9497591,True
3,rows_in_minus_removed,9497591,True


In [25]:
tidy.to_parquet(PROCESSED_DIR / "supermarket_prices_tidy.parquet", index=False)

product_lookup.to_csv(PROCESSED_DIR / "product_lookup.csv", index=False)

reconciliation.to_csv(
    RESULTS_DIR / "cleaning_reconciliation.csv",
    index=False
)

print("Saved tidy dataset:", PROCESSED_DIR / "supermarket_prices_tidy.parquet")
print("Saved product lookup:", PROCESSED_DIR / "product_lookup.csv")
print("Saved reconciliation:", RESULTS_DIR / "cleaning_reconciliation.csv")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [26]:
tidy.to_csv(PROCESSED_DIR / "supermarket_prices_tidy.csv", index=False)

product_lookup.to_csv(PROCESSED_DIR / "product_lookup.csv", index=False)

reconciliation.to_csv(
    RESULTS_DIR / "cleaning_reconciliation.csv",
    index=False
)

print("Saved tidy dataset:", PROCESSED_DIR / "supermarket_prices_tidy.csv")
print("Saved product lookup:", PROCESSED_DIR / "product_lookup.csv")
print("Saved reconciliation:", RESULTS_DIR / "cleaning_reconciliation.csv")

Saved tidy dataset: /Users/charan/Documents/supermarket-price-direction/data/processed/supermarket_prices_tidy.csv
Saved product lookup: /Users/charan/Documents/supermarket-price-direction/data/processed/product_lookup.csv
Saved reconciliation: /Users/charan/Documents/supermarket-price-direction/results/data_cleaning/cleaning_reconciliation.csv


In [27]:
print("Tidy rows:", len(tidy))
print("Tidy columns:", tidy.columns.tolist())

print("Missing product_id:", tidy["product_id"].isna().sum())
print("Missing retailer:", tidy["retailer"].isna().sum())
print("Missing date:", tidy["date"].isna().sum())

duplicate_final_check = tidy.duplicated(
    subset=["product_id", "retailer", "date"],
    keep=False
).sum()

print("Final duplicate product_id-retailer-date rows:", duplicate_final_check)

Tidy rows: 9497591
Tidy columns: ['product_id', 'retailer', 'category', 'date', 'product_name', 'product_name_normalised', 'price', 'price_per_unit', 'unit', 'own_brand_flag', 'capture_gap_before_row', 'previous_capture_date', 'days_since_previous_capture']
Missing product_id: 0
Missing retailer: 0
Missing date: 0
Final duplicate product_id-retailer-date rows: 0


In [28]:
cleaning_decision_log = pd.DataFrame([
    {
        "decision": "Duplicate key resolution",
        "rule": "For duplicate product_id-retailer-date keys, keep the row with the highest completeness score; if tied, keep the earliest raw row.",
        "rows_affected": len(duplicate_drop_log),
        "reason": "A single product-retailer-date observation is required for time-series modelling."
    },
    {
        "decision": "Product-name normalisation",
        "rule": "Create product_name_normalised using lowercase text, punctuation removal, and whitespace collapse; keep original product_name and save mapping.",
        "rows_affected": len(product_name_mapping),
        "reason": "Product identifiers must be stable while preserving the original raw names."
    },
    {
        "decision": "Capture gaps",
        "rule": "Flag rows where the previous observation for the same product-retailer series is more than one day earlier; do not delete rows.",
        "rows_affected": int(tidy["capture_gap_before_row"].sum()),
        "reason": "Later lag and rolling features must avoid silently spanning missing capture periods."
    },
    {
        "decision": "Final reconciliation",
        "rule": "Rows out must equal rows in minus rows dropped as duplicate keys.",
        "rows_affected": len(tidy),
        "reason": "Cleaning must be auditable and row counts must reconcile exactly."
    }
])

cleaning_decision_log.to_csv(
    RESULTS_DIR / "cleaning_decision_log.csv",
    index=False
)

cleaning_decision_log

,decision,rule,rows_affected,reason
0,Duplicate key resolution,"For duplicate product_id-retailer-date keys, k...",31651,A single product-retailer-date observation is ...
1,Product-name normalisation,Create product_name_normalised using lowercase...,130633,Product identifiers must be stable while prese...
2,Capture gaps,Flag rows where the previous observation for t...,655945,Later lag and rolling features must avoid sile...
3,Final reconciliation,Rows out must equal rows in minus rows dropped...,9497591,Cleaning must be auditable and row counts must...
